# Earnalism music job `music-a45f086f`

**Prepared, not run.** Input zip SHA256 `ac71cab4d851c8730c133ff73046cad66f784e7abe65ee959dc052a309a7c94b`. Copy `inputs.zip` to `/content/drive/MyDrive/earnalism-audiovisual/kshudhita-pashan/music-a45f086f/` in Drive, set `RUN = True` in the first cell, then run all. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 2760 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'music-a45f086f'
TITLE = 'kshudhita-pashan'
KIND = 'music'
INPUT_ZIP_SHA256 = 'ac71cab4d851c8730c133ff73046cad66f784e7abe65ee959dc052a309a7c94b'
MODEL_REPO = 'ACE-Step/ACE-Step-v1-3.5B'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 60
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/kshudhita-pashan/music-a45f086f'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 2760
MANIFEST_RESERVE_S = 60
PACKAGES = ('ace-step', 'acestep', 'torch', 'transformers', 'diffusers', 'huggingface_hub')


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import threading, time
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, runtime.unassign)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    """Between items: start the next one only if it (and the manifest) fits before the hard stop."""
    global INCOMPLETE
    left = DEADLINE_SECONDS - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s forecast; no further item starts" % (left, DEADLINE_SECONDS))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Music: ACE-Step. The exact commit and weight hashes are resolved now and recorded.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/ace-step/ACE-Step.git"], check=True)
from huggingface_hub import HfApi, snapshot_download

revision = HfApi().model_info(MODEL_REPO).sha
weights_dir = snapshot_download(MODEL_REPO, revision=revision)


def _sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


weight_hashes = {}
for p in sorted(pathlib.Path(weights_dir).rglob("*")):
    if p.is_file() and p.suffix in (".safetensors", ".bin", ".pt", ".pth"):
        weight_hashes[p.relative_to(weights_dir).as_posix()] = _sha(p)
MODEL_INFO = {"repo": MODEL_REPO, "revision": revision, "weight_sha256": weight_hashes}

from acestep.pipeline_ace_step import ACEStepPipeline

pipe = ACEStepPipeline(checkpoint_dir=weights_dir, dtype="bfloat16")
EXPECTED = [t["key"] + ".wav" for t in JOB["items"]]
last = 0.0  # seconds the previous cue took; the next starts only if it fits (R25)
for t in JOB["items"]:
    if not may_start(last):
        break
    started = time.time()
    pipe(prompt=t["prompt"], lyrics=t.get("lyrics", "[instrumental]"),
         audio_duration=t["duration_s"], infer_step=60, guidance_scale=15.0,
         manual_seeds=[t["seed"]], save_path=str(OUT_DIR / (t["key"] + ".wav")))
    last = time.time() - started
    print("wrote", t["key"], "in %.1f s" % last)


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file() and p.name not in ("manifest.json", "inputs.zip"):
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(time.time() - JOB_STARTED, 1),
            "packages": packages, "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
